# Cardiovascular Risk Calculation Model

## 1. Model Overview & Input Weights

| Risk Input Factor | MI Beta Coefficient | Stroke Beta Coefficient |
| :--- | :--- | :--- |
| **Age** | 0.0719 | 0.0987 |
| **Total Cholesterol (TC)** | 0.2285 | 0.0295 |
| **Systolic Blood Pressure (SBP)** | 0.0132 | 0.0223 |
| **Diabetes** | 0.6410 | 0.6269 |
| **Smoking** | 0.5638 | 0.4981 |
| **TC x Age Interaction** | -0.0046 | 0.0014 |
| **SBP x Age Interaction** | -0.0002 | -0.0004 |
| **Diabetes x Age Interaction** | -0.0125 | -0.0263 |
| **Smoking x Age Interaction** | -0.0183 | -0.0151 |

---

## 2. Baseline Survival Probabilities
* **MI Baseline Survival:** `0.9540`
* **Stroke Baseline Survival:** `0.9849`

---

## 3. Patient Benchmark Results

| Patient | Age | Total Chol | SBP | Diabetes | Smoking | MI 10-Yr Risk | Stroke 10-Yr Risk | Combined CVD Risk |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Person 1** | 35 | 3.103 | 105.5 | No (0) | No (0) | **0.22%** | **0.08%** | **0.30%** |
| **Person 2** | 41 | 2.585 | 118.0 | No (0) | No (0) | **0.39%** | **0.22%** | **0.61%** |
| **Person 3** | 62 | 4.266 | 134.0 | No (0) | No (0) | **4.35%** | **2.34%** | **6.59%** |

## 4. Stata Execution Code
```stata
cd "C:\Users\ak1730\OneDrive - Mass General Brigham Incorporated\Desktop"
import delimited "HPACC_Oldwaves_2025-07-15.csv", clear varnames(1)
keep in 1/50
gen csmoke_num = 1 if csmoke=="Yes"
replace csmoke_num = 0 if csmoke=="No"
drop csmoke
rename csmoke_num csmoke

local age_cen  60
local tc_cen   6
local sbp_cen  120

gen mi_sumbeta = 0.0719*(age-`age_cen') + 0.2285*(tchol_mmoll-`tc_cen') + 0.0132*(sbp_avg-`sbp_cen') + 0.6410*clin_dia + 0.5638*csmoke - 0.0046*(age-`age_cen')*(tchol_mmoll-`tc_cen') - 0.0002*(age-`age_cen')*(sbp_avg-`sbp_cen') - 0.0125*(age-`age_cen')*clin_dia - 0.0183*(age-`age_cen')*csmoke
gen mi_10yr_risk = 1 - (0.9540)^exp(mi_sumbeta)

gen stroke_sumbeta = 0.0987*(age-`age_cen') + 0.0295*(tchol_mmoll-`tc_cen') + 0.0223*(sbp_avg-`sbp_cen') + 0.6269*clin_dia + 0.4981*csmoke + 0.0014*(age-`age_cen')*(tchol_mmoll-`tc_cen') - 0.0004*(age-`age_cen')*(sbp_avg-`sbp_cen') - 0.0263*(age-`age_cen')*clin_dia - 0.0151*(age-`age_cen')*csmoke
gen stroke_10yr_risk = 1 - (0.9849)^exp(stroke_sumbeta)

gen cvd_10yr_risk = 1 - (1 - mi_10yr_risk) * (1 - stroke_10yr_risk)
list country age tchol_mmoll sbp_avg clin_dia csmoke mi_10yr_risk stroke_10yr_risk cvd_10yr_risk, clean
```